In [43]:
import os
import pandas as pd
import numpy as np

from syndat.metrics import (
    jensen_shannon_distance,
    normalized_correlation_difference,
    discriminator_auc
)
from dotenv import load_dotenv
from typing import Literal

import pyinputplus as pyip
import pprint
import textwrap
import json
from pandas.api.types import CategoricalDtype

from pathlib import Path
import subprocess as sp

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool, ToolRuntime # can provide a description or name of function
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from sklearn.impute import SimpleImputer

load_dotenv()
data_path = "../data/cat_texas.csv" # PUDF_base1_1q2019 (mostly categorical)
#data_path = "../data/num_iris.csv" # (mostly numerical)
#data_path = "../data/mix_titanic.csv" # (mix of categorical and numerical)


def format_response(text:str):
    print(textwrap.fill(text, width=100))

# Note: usage_metadata could later be used to assess prompting and the workflow

In [44]:
# quick setup
synthesis_request = {'data_scope': {'selected_columns': []},
 'generator': {'name': 'auto'},
 'missing_value_handling': {'impute': {'categorical_strategy': 'mode',
                                       'method': 'SimpleImputer',
                                       'numerical_strategy': 'mean'},
                            'mode': 'impute',
                            'threshold': {'set_threshold': None}},
 'privacy': {'mode': 'standard', 'sensitive_columns': []},
 'sample_size': {'mode': 'fixed value', 'value': 10},
 'synthesis_goal': {'goal': 'basic_generation',
                    'target_column': None,
                    'task': None}}

#df.to_csv("../data/cat_texas_stypes.csv", index=False)
df = pd.read_csv("../data/cat_texas_dtypes.csv")
# again, semantic types can be used for quick retracing of steps.

def match_to_semantic_types(df, semantic_types):
    df = df.copy()
    for col, semantic_type in semantic_types.items():
        if semantic_type == "Integer":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

        elif semantic_type == "Float":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

        elif semantic_type == "Categorical":
            df[col] = df[col].astype("string").astype("category")

        elif semantic_type == "Ordinal":
            dtype = CategoricalDtype(categories=df[col].dropna().unique(), ordered=True)
            df[col] = df[col].astype("string").astype(dtype)
    return df

# df.to_csv("../data/cat_texas_dtypes.csv", index=False)
# save point
df = pd.read_csv("../data/cat_texas_dtypes.csv")

semantic_types = {'RECORD_ID': 'Integer',
 'TYPE_OF_ADMISSION': 'Categorical',
 'SOURCE_OF_ADMISSION': 'Categorical',
 'PUBLIC_HEALTH_REGION': 'Categorical',
 'SEX_CODE': 'Categorical',
 'RACE': 'Categorical',
 'ETHNICITY': 'Categorical',
 'ADMIT_WEEKDAY': 'Categorical',
 'LENGTH_OF_STAY': 'Integer',
 'PAT_AGE': 'Categorical',
 'FIRST_PAYMENT_SRC': 'Categorical',
 'ADMITTING_DIAGNOSIS': 'Categorical',
 'PRINC_DIAG_CODE': 'Categorical',
 'POA_PRINC_DIAG_CODE': 'Categorical'}

df = match_to_semantic_types(df, semantic_types)

#### Generate dataset report:

In [45]:
def select_preferences() -> dict:
    cols_of_interest = [] # if empty, all columns should be included
    synthesis_request = {
        "synthesis_goal": {
            "goal": "basic_generation",  
            # TODO: change predictive utility to the format below
            # "predictive_utility": {
            #     "task": None, # regression, classification, survival, None
            #     "target_column": None,
            # }
            "task": None,
            "target_column": None,
        },
        # Q: Which columns should be included?
        "data_scope": {
            "selected_columns": cols_of_interest # If empty, all cols
            },
        "sample_size": {
            "mode": "fixed", # fixed, fraction, same_as_original
            "value": 10, # just for testing workflow
            # TODO change to this structure
            # "fixed": {
            #     "value": None,
            # },
            # "fraction": {
            #     "value": None,
            # },
        },
        "privacy": {
            "mode": "standard", # standard, differentially private
            # TODO change to this structure
            # "differentially_private": {
            #     "sensitive_columns": [],
            # }
            "sensitive_columns": [],
        },
        "generator": {
            "name": "auto", # Recommend data generator
            #"parameters": {}, # Depends on generators used/available (not important for right now)
        },
        "missing_value_handling": {
            "mode": "auto", # impute, threshold, remove
            "impute": {
                "method": None, # SimpleImputer
                "categorical_strategy": None, # auto, most_frequent, constant
                "numerical_strategy": None, # auto, mean, median, most_frequent
            },
            "threshold": {
                "set_threshold": None, # minimum number of missing columns in a sample to disregard
            }
        }
        # Possibly a note for the evaluation type as well?
    }

    # Fill synthesis goal
    print("What would you like to use the synthetic data for?")

    choice = pyip.inputMenu(["basic generation", "predictive utility"], numbered=True)
    if choice == "predictive utility":
        synthesis_request["synthesis_goal"]["goal"] = "predictive_utility"
        print("\nWhich downstream machine learning task are you interested in?")
        
        choice = pyip.inputMenu(["classification", "regression", "survival"], numbered=True) # default none
        synthesis_request["synthesis_goal"]["task"] = choice
        print("\nWhat is the target column? Select 'auto' if you would like a suggestion.")
        # TODO think about how to include column names in an easier way
            # I can think of giving an all selection option, but it is very annoying to individually
            # and perfectly type in all column names to be accepted.
        #target = pyip.inputChoice([],"If applicable, what is the target column?") # one entry or auto?
        #synthesis_request["synthesis_goal"]["target_column"] = target 

    # Fill data_scope
    #response = pyip.inputYesNo("Are there any columns in the dataset that you would like to exclude from generation?")
        # Q: Which columns should be included?
        # TODO add above questoin as prompt
    synthesis_request["data_scope"]["selected_columns"] = cols_of_interest

    # Fill sample size
    print("\nWould you like to generate the same amount of samples as the dataset or a fixed number of samples?")

    choice = pyip.inputMenu(["fixed value", "same as original"], numbered=True) # fixed, same_as_original, fraction_of_original
    synthesis_request["sample_size"]["mode"] = choice
    
    if choice == "fixed value":
        print("\nHow many synthetic samples would you like to generate?")

        value = pyip.inputInt(min=1)
        synthesis_request["sample_size"]["value"] = value
    
    # possible TODO: incorporate an option to for user to input a fraction of the dataset.

    response = pyip.inputYesNo("\nAre you interested in differential privacy?")
    if response == "yes":
        synthesis_request["privacy"]["mode"] = "differential_privacy"
        # TODO: check if there are any sensitive columns (what is a way to include columns that is not cumbesome?)

    # Fill data generator preferences
    response = pyip.inputYesNo("\n\nIs there a specific data generator that you want to use on your data?")
    if response == "yes":
        choice = pyip.inputMenu(["CTGAN", "PATEGAN", "BaysianNet",])
        synthesis_request["generator"]
    else:
        print("\n\tYou will receive a recommendation based on the dataset's characteristics.")
        # check that the data generator is available for use (probably ctgan, pategan, and tvae)
        # check parameter preferences of the user. Depends on generators used/available
        
    # Missing value handling preferences
    print("\nHow would you like missing values to be handled? Select 'auto' if you would like a suggestion.")
    choice = pyip.inputMenu(["impute", "threshold", "remove", "auto"], numbered=True)
    if choice == "auto":
        synthesis_request["missing_value_handling"]["mode"] = choice

    elif choice == "threshold":
        synthesis_request["missing_value_handling"]["mode"] = choice
        value = pyip.inputInt("\nWhat is the minimum number of missing columns in a sample remove from the data?")
        synthesis_request["missing_value_handling"][choice]["set_threshold"] = value

    elif choice == "remove":
        synthesis_request["missing_value_handling"]["mode"] = choice

    elif choice == "impute":
        synthesis_request["missing_value_handling"]["mode"] = choice
        print("\nOnly one imputation method currently exists. SimpleImputer will be selected.")
        synthesis_request["missing_value_handling"][choice]["method"] = "SimpleImputer"
        # TODO Add additional imputers
        
        print("\nHow would you like to handle missing categorical values? Select 'auto' if you would like a suggestion.")
        choice = pyip.inputMenu(["auto", "mode", "constant"], numbered=True)
        if choice == "mode": # Changed externally to be a bit more understandable: mode vs most_frequent
            synthesis_request["missing_value_handling"]["impute"]["categorical_strategy"] = "most_frequent"
        synthesis_request["missing_value_handling"]["impute"]["categorical_strategy"] = choice

        print("\nHow would you like to handle missing numerical values? Select 'auto' if you would like a suggestion.")
        choice = pyip.inputMenu(["auto", "mean", "median", "mode"], numbered=True)
        if choice == "mode":
            synthesis_request["missing_value_handling"]["impute"]["numerical_strategy"] = "most_frequent"
        synthesis_request["missing_value_handling"]["impute"]["numerical_strategy"] = choice
 
    print()
    print("-"*25+"Your preferences have been recorded"+"-"*25)
    pprint.pprint(synthesis_request)
    return synthesis_request

#synthesis_request = select_preferences()

In [46]:
# Moved the run_synthetic_generator tool inside for the model to only be able 
# to choose the mechanism (synthetic generator)
def make_generator_tool(data_path:str, synthesis_request:dict):
    @tool("run_synthetic_generator", description="Run the selected synthetic data generator.", return_direct=False) # return direct have tool immediately idrect ot user
    def run_synthetic_generator(mechanism:Literal["PATEGAN", "CTGAN", "BayesianNet", "PrivBayes",]):
        # Set sample size
        if synthesis_request["sample_size"]["mode"] == "fixed value":
            sample_size = synthesis_request["sample_size"]["value"]
        else:
            sample_size = pd.read_csv(data_path).shape[0]

        # Set cleaned data and output paths
        data_path_name = Path(data_path).stem
        output_filename = f"synthetic_{data_path_name}_{mechanism.lower()}.csv"
        output_path = f"../output/{output_filename}"
        
        print(f"Running {mechanism}")

        command = f"""
        podman run -it --rm \
            --entrypoint /bin/bash -v "$PWD/..:/workspace:Z" \
            docker.io/springepfl/synthetic-data:latest \
            -c "cd /synthetic_data_release && python3 executables/generate_synthetic_dataset.py \
            -D /workspace/data/{data_path_name} \
            -M {mechanism} \
            -O /workspace/output/{output_path}\
            -N {sample_size}"
            """
        process = sp.Popen(
            command,
            shell=True,
            stdout=sp.PIPE,
            stderr=sp.STDOUT,
            text=True
            )

        for line in process.stdout:
            print(line, end="")

        process.wait()
        
        # If process fails for a specific generator
        if process.returncode !=0:
            raise RuntimeError(f"{mechanism} failed.")

        print(f"\n{mechanism}: {sample_size} synthetic samples successfully generated.")
        return output_path
    
    return run_synthetic_generator

generator_tool = make_generator_tool('../data/cat_texas_after_imputation.csv', synthesis_request)

# Load agent
model = init_chat_model(
    "MiniMaxAI/MiniMax-M2.5",
    model_provider="openai",
    base_url=os.environ.get("BASE_URL"),
)

checkpointer = InMemorySaver() #for remembering conversations (can add invocation id)
config = {'configurable': {'thread_id': 1}}

agent = create_agent(
    model=model,
    tools=[generator_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                generator_tool.name: {
                    "allowed_decisions": ["approve", "edit", "reject", "respond"]
                }
            },
            description_prefix="Synthetic generator execution pending approval.",
        )
    ],
    #tools = [get_weather, locate_user],
    #system_prompt='You are a helpful weather assistant, who always cracks jokes and is humorous while remaining helpful.',
    checkpointer=checkpointer, # add memory to the model to keep track of convos
)

1. Upload data (verify valid format for pandas)

In [47]:
def upload_data(upload_path:str, selected_columns:list) -> pd.DataFrame:
    """
    Verify uploaded data can be made into a pandas dataframe.
    Currently only tested for one file, would need to be updated to
    account for multiple files (synthcraft has a multiple file uploader
    which might be worth looking into)
    """
    path = upload_path.lower()

    # if path.endswith(".xlsx"):
    # If this shoudl be added, needs some for cols that are not selected
    #     df = pd.read_excel(upload_path, usecols=selected_columns)
    #     if df.shape[0] > 1 and df.shape[1] > 1:
    #         raise ValueError("Invalid entry. Verify excel file is in a tidy format (example: https://aeturrell.github.io/python4DS/data-tidy.html).")
    #     return df

    for sep in [",", "\t", ";", "|"]:
        try:
            if len(selected_columns) == 0:
                df = pd.read_csv(upload_path, delimiter=sep)
                if df.shape[0] > 1 and df.shape[1] > 1:
                    return df
            else:
                df = pd.read_csv(upload_path, usecols=selected_columns, delimiter=sep)
                if df.shape[0] > 1 and df.shape[1] > 1:
                    return df
            
        except Exception:
            continue

    raise ValueError("Invalid entry; File type may not be supported. Verify file is in a tidy format (example: https://aeturrell.github.io/python4DS/data-tidy.html).")


#df = upload_data(data_path, synthesis_request["data_scope"]["selected_columns"])

2. Clean deterministically

a) Handle data types

In [48]:
# Maybe could be improved so that multiple columns can be assigned to a specific dtype at once?
def infer_dtype(series):
    """Infers pandas storage dtypes by taking the majority type of each column"""
    non_missing = series.dropna()

    if len(non_missing) == 0:
        return "string"
    
    type_counts = series.dropna().apply(lambda x: type(x).__name__).value_counts(normalize=True)
    majority_type = type_counts.idxmax()

    if majority_type == "str":
        return "string"

    elif majority_type == "int":
        return "Int64"
    
    elif majority_type == "float":
        whole_numbers = ((non_missing % 1) == 0).all()
        if whole_numbers:
            return "Int64"
        return "float64"
    
    # TODO: add additional data types datetime
    return "string"


# TODO Consider what to do with Datetime values (have to likely test with the tool if it can handle that)

# Check for unofficial missing values which can initially be encoded as NA
# Show data types present in each column.
# Try to match dtypes to the majority present in each column. 
# (not of concern for right now) If there is a larger issues with a column, possibly employ agent to assist with cleaning the data
# Show updated data types present in each column (if there was a change)
# Confirm with user that the data types are correct
# if any are incorrect, have user specify the column (possibly one  by one) or might be easier to add an agent if multiple columns are affected


def handle_dtypes(df:pd.DataFrame):
    # Replace unofficial missing values
    confirm = pyip.inputYesNo("Are there any missing values denoted by a character/value other than NA (e.g., '?', '`', 'NA')?")
    if confirm == "yes":
        # TODO maybe incorportae a check if the value is a string or int/float so it can be represented.
        value = input("Input the value that will replace all equivalent entries in the dataset:")
        total = (df == value).sum().sum()

        if total == 0:
            print(f"\nThere are none of the specified value ({value}) in the dataset.")
        else:
            print(f"\nThere are {total} total values that will be replaced with NA in the dataset.")
            confirm = pyip.inputYesNo(f"Should the following be replaced with NA?: {value}")
            if confirm == "yes":
                df = df.replace(value, np.nan)
                print(f"\nThe following entry has been replaced with NA in the dataset: {value}")
            else:
                print(f"\nThe following entry has not been replaced with NA in the dataset: {value}")
                # possible TODO: Add attempting another input?

    # General check for majority int string or float and convert all in the column main dtype
    for col in df.columns:
        found_dtype = infer_dtype(df[col])
        if found_dtype == "string":
            df[col] = df[col].astype("string")
        # Possibly could set errors to default (raise), but would have to try to handle this.
        elif found_dtype == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif found_dtype == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    
    print(f"\nCurrent data types present in each column:\n{df.dtypes}")
    print("\nSupported data types are as follows: Integer, Float, and String.")
    confirm = pyip.inputYesNo("Do any of the columns need a correction?")
    # TODO: Add a way to select multiple columns and convert them to a specific datatype all together.
    if confirm == "yes":
        while True:
            print(f"\nInput column name/number to modify (enter exit (not case-sensitive) or 1 to finish)")
            var = pyip.inputMenu(["Exit"] + df.columns.tolist(), numbered=True)

            if var == "Exit":
                break

            if var not in df.columns:
                print("Invalid column name.")
                continue

            print(f"Selected the following column: {var}")
            choice = pyip.inputMenu(["Integer", "Float", "String",], numbered=True) 
            
            if choice == "Integer":
                df[var] = pd.to_numeric(df[var], errors="coerce").astype("Int64")
            
            elif choice == "Float":
                df[var] = pd.to_numeric(df[var], errors="coerce").astype("float64")

            elif choice == "String":
                df[var] = df[var].astype("string")

            print(f"\nUpdated dtype for {var}: {df[var].dtype}")
            print(f"For reference: Data types present in each column:\n{df.dtypes}")
            print("-"*40)
    print()
    print("-"*25+"Final data types present in each column"+"-"*25)
    print(df.dtypes)
    return df

#df = handle_dtypes(df)
# TODO: Add some sort of save (think about how to store for easy retrieval)
# maybe saving and storing the path of each resulting dataframe after each block 
# as a dictionary of lists where msot recent entries are at the end fo the lists (but that way, everything can be easily retrived)

In [49]:
def handle_semantic_types(df):
    semantic_types = {}
    for col in df:
        if pd.api.types.is_integer_dtype(df[col]):
            semantic_types[col] = "Integer"
        elif pd.api.types.is_float_dtype(df[col]):
            semantic_types[col] = "Float"
        elif pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("category")
            semantic_types[col] = "Categorical"

    print("View of the rudimentary semantic types found for each column:")
    for col in df.columns:
        print(f"{col}: {semantic_types[col]}")

    print("\nSupported semantic types are as follows: Integer, Float, Categorical, and Ordinal.")
    confirm = pyip.inputYesNo("Do any of the columns need additional specification?")
    # TODO: Add a way to select multiple columns and convert them to a specific datatype all together.
    if confirm == "yes":
        while True:
            print(f"\nInput column name/number to modify (enter exit (not case-sensitive) or 1 to finish)")
            var = pyip.inputMenu(["Exit"] + df.columns.tolist(), numbered=True)

            if var == "Exit":
                break

            if var not in df.columns:
                print("Invalid column name.")
                continue

            print(f"Selected the following column: {var}")
            choice = pyip.inputMenu(["Integer", "Float","Categorical", "Ordinal"], numbered=True) 
            
            if choice == "Integer":
                df[var] = pd.to_numeric(df[var], errors="coerce").astype("Int64")
                semantic_types[var] = choice
            
            elif choice == "Float":
                df[var] = pd.to_numeric(df[var], errors="coerce").astype("float64")
                semantic_types[var] = choice

            elif choice == "Categorical":
                df[var] = df[var].astype("category")
                semantic_types[var] = choice

            elif choice == "Ordinal":
                dtype = CategoricalDtype(categories=df[var].dropna(), ordered=True)
                df[var] = (df[var].astype(str).astype(dtype))
                semantic_types[var] = choice
            print(f"\nUpdated semantic type for {var}: {semantic_types[var]}")
            print(f"For reference: Semantic types present in each column:")
            for col in df.columns:
                print(f"{col}: {semantic_types[col]}")
            print("-"*40)
    print()
    print("-"*25+"Final data types present in each column"+"-"*25)
    for col in df.columns:
        print(f"{col}: {semantic_types[col]}")
    return df, semantic_types
#df, semantic_types = handle_semantic_types(df)

b) Handle missing values

In [58]:
def update_missing_value_choice(synthesis_request: dict):
    print("Please select another method to handle missing values:")

    choice = pyip.inputChoice(["impute", "threshold", "auto"])
    synthesis_request["missing_value_handling"]["mode"] = choice

    # If any additional specifications are required
    if choice == "threshold":
        value = pyip.inputInt("Set the maximum number of missing values allowed per row:", min=0)
        synthesis_request["missing_value_handling"][choice]["set_threshold"] = value

    elif choice == "impute":
        #method = pyip.inputChoice(["NA"], prompt="Please select an imputation method:")
        synthesis_request["missing_value_handling"][choice]["method"] = "SimpleImputer"
        print("\nHow would you like to handle missing categorical values? Select 'auto' if you would like a suggestion.")
        choice = pyip.inputMenu(["auto", "mode", "constant"], numbered=True)
        if choice == "mode": # Changed externally to be a bit more understandable: mode vs most_frequent
            synthesis_request["missing_value_handling"]["impute"]["categorical_strategy"] = "most_frequent"
        synthesis_request["missing_value_handling"]["impute"]["categorical_strategy"] = choice

        print("\nHow would you like to handle missing numerical values? Select 'auto' if you would like a suggestion.")
        choice = pyip.inputMenu(["auto", "mean", "median", "mode"], numbered=True)
        if choice == "mode":
            synthesis_request["missing_value_handling"]["impute"]["numerical_strategy"] = "most_frequent"
        synthesis_request["missing_value_handling"]["impute"]["numerical_strategy"] = choice

    return synthesis_request


def update_missing_handling_plan(synthesis_request:dict, missing_plan:dict):
    update = synthesis_request["missing_value_handling"].copy()

    update["mode"] = missing_plan["mode"]

    update["impute"]["method"] = missing_plan["imputation_method"]
    update["impute"]["categorical_strategy"] = missing_plan["categorical_strategy"]
    update["impute"]["numerical_strategy"] = missing_plan["numerical_strategy"]

    update["threshold"]["set_threshold"] = missing_plan["set_threshold"]

    synthesis_request["missing_value_handling"] = update

    return synthesis_request


def handle_missing_values(df:pd.DataFrame, synthesis_request:dict):
    if df.isna().sum().sum() == 0:
        print("There are no missing values in the dataset.")
    else:
        #TODO add an initial check for missing values
        mode = synthesis_request["missing_value_handling"]["mode"]
        # 1. Remove all samples with NA columns
        if mode == "remove":
            null_row_count = df.isna().any(axis=1).sum()
            print(f"Warning: {null_row_count} out of {df.shape[0]} rows ({null_row_count/df.shape[0]*100}%) will be removed from the dataset.")
            confirm = pyip.inputYesNo("Confirm whether this should be done (yes/no/y/n):")
            if confirm == "yes":
                df = df.dropna()
                return df, synthesis_request
            else:
                # If this option is too harsh and the user wants to try another method.
                synthesis_request = update_missing_value_choice(synthesis_request)
                return handle_missing_values(df, synthesis_request)
            
        # 2. Remove all samples above a specified threshold of NA columns 
        elif mode == "threshold":
            threshold = synthesis_request["missing_value_handling"]["threshold"]["set_threshold"]
            # TODO add a limit to the threshold such that it doesn't add mroe than
            threshold_samples = df.isna().sum(axis=1) > threshold
            count = threshold_samples.sum()

            print(f"Warning: {count} out of {df.shape[0]} ({count/df.shape[0]*100}%) will be removed from the dataset.")
            confirm = pyip.inputYesNo("Confirm whether this should be done.")
            if confirm == "yes":
                df = df[~threshold_samples]
                return df, synthesis_request
            else:
                confirm = pyip.inputYesNo("Would you like to select a new threshold value?")
                if confirm == "yes":
                    value = pyip.inputInt("Enter a new threshold value.", min=1) # Double check how I have it written
                    synthesis_request["missing_value_handling"]["set_threshold"] = value
                    return handle_missing_values(df, synthesis_request)
                    
                synthesis_request = update_missing_value_choice(synthesis_request)
                # TODO add a limit to the threshold such that it doesn't add more
                #  possible TODO allow for threshold changes (if found to be too harsh but want to stay with a threshold)

                return handle_missing_values(df, synthesis_request)

        elif mode == "impute":
            # Separate user defined categorical and numerical values for imputation
            cat_cols = df.select_dtypes(include="category").columns.to_list()
            num_cols = df.select_dtypes(include="number").columns.to_list()

            # Categorical value imputation
            cat_strategy = synthesis_request["missing_value_handling"]["impute"]["categorical_strategy"]
            if cat_cols:
                cat_data = df[cat_cols].astype("object").where(df[cat_cols].notna(), np.nan)
                if cat_strategy == "constant":
                    print("\nEnter the fill value that missing values should have. Fill values must be strings.\nPress enter if you would like to stay with the default ('Unknown'):")
                    fill_value = pyip.inputStr()
                    if fill_value == "":
                        fill_value = "Unknown"
                        imputer = SimpleImputer(strategy=cat_strategy, fill_value=fill_value)
                elif cat_strategy == "mode":
                    imputer = SimpleImputer(strategy="most_frequent")
                else:
                    imputer = SimpleImputer(strategy=cat_strategy)

                df[cat_cols] = pd.DataFrame(imputer.fit_transform(cat_data),columns=cat_cols, index=df.index)
                print(f"\nThere are {df[cat_cols].isna().sum().sum()} remaining null values in the categorical columns.")

            # Numerical value imputation
            if num_cols:
                num_strategy = synthesis_request["missing_value_handling"]["impute"]["numerical_strategy"]
                imputer = SimpleImputer(strategy=num_strategy)
                df[num_cols] = pd.DataFrame(imputer.fit_transform(df[num_cols]), columns=df[num_cols].columns, index=df.index)
                print(f"\nThere are {df[num_cols].isna().sum().sum()} remaining null values in the categorical columns.")

            print(f"\nThere are {df.isna().sum().sum()} remaining null values in the dataset.")
            df = match_to_semantic_types(df, semantic_types)
            return df, synthesis_request

        # 5. Use an LLM to suggest the missing value handling strategy
        if mode == "auto":
            proportion_missing = (df.isna().sum() / df.shape[0]).round(4).to_dict()
            rows_with_missing = int(df.isna().any(axis=1).sum())

            prompt = ChatPromptTemplate.from_messages([
                ("system", """
You are ONLY advising on missing value handling for synthetic tabular data preprocessing.

Always return exactly one valid JSON object.
No markdown. No text outside the JSON.
    
Allowed modes:
- remove
- threshold
- impute

Allowed imputation_method values:
- SimpleImputer
    
Allowed categorical_strategy values:
- mode
- constant

Allowed numerical_strategy values:
- mean 
- median
- mode 

Rules:
- If any setting has "auto", this indicates the user wants your help to determine the optimal settings to use.
- If the mode value has a key of the same name (impute, threshold), all entries within that key must be filled.
- If there are no missing values, recommend mode "remove" and explain that no rows will be removed.
- If few rows have missing values, recommend "remove".
- If many rows have missing values but only some rows are heavily incomplete, recommend "threshold".
- If missing values are spread across many rows, recommend "impute".
- Always include a user-facing plan and rationale.

JSON format:
{{
"missing_value_handling": {{
    "mode": "remove",
    "set_threshold": null,
    "imputation_method": null,
    "categorical_strategy": "mode",
    "numerical_strategy": "mean",
    "columns_to_remove": [],
    "needs_user_confirmation": true,
    "rationale": "Brief explanation."
}}
}}
            """),
                ("user", """
            Missing value proportions per column:
            {proportion_missing}

            Rows with at least one missing value:
            {rows_with_missing}

            Total rows:
            {total_rows}

            Current synthesis request:
            {synthesis_request}
                
            Semantic types of each column:
            {semantic_types}
                
            Plan update from user, if any:
            {plan_update}
            """)
            ])

            plan_update = None
            while True:
                chain = prompt | agent
                plan = chain.invoke({
                    "proportion_missing": proportion_missing,
                    "rows_with_missing": rows_with_missing,
                    "total_rows": df.shape[0],
                    "synthesis_request": synthesis_request,
                    "semantic_types": semantic_types,
                    "plan_update": plan_update,
                    },
                    config = config,
                    )
                plan = plan['messages'][-1].content.split("</think>")[1].replace("\n", "")
                parsed = json.loads(plan)
                missing_plan = parsed["missing_value_handling"]

                print("Suggested plan:")
                print(f"- Mode: {missing_plan['mode']}")
                print(f"- Threshold: {missing_plan['set_threshold']}")
                print(f"- Imputation method: {missing_plan['imputation_method']}")
                print(f"- Categorical strategy: {missing_plan['categorical_strategy']}")
                print(f"- Numerical strategy: {missing_plan['numerical_strategy']}")

                if len(missing_plan["columns_to_remove"]) > 0:
                    print(f"- Columns to remove: {missing_plan["columns_to_remove"]}")
                print(f"\nReasoning behind the suggestion:")
                format_response(missing_plan["rationale"])

                confirm = pyip.inputYesNo("\nWould you like to proceed with the suggestion?")
                if confirm == "yes":
                    synthesis_request.update({
                        "mode": missing_plan["mode"],
                        "set_threshold": missing_plan["set_threshold"],
                        "imputation_method": missing_plan["imputation_method"],
                        "categorical_strategy": missing_plan["categorical_strategy"],
                        "numerical_strategy": missing_plan["numerical_strategy"],
                        "columns_to_remove": missing_plan["columns_to_remove"],
                        })
                    print(f"\n\nThe missing value handling plan has been updated to the following:")
                    pprint.pprint(synthesis_request)
                    break

                else:
                    print("\nPlease describe how you would want missing values to be handled and what changes you would make to the suggested plan.")
                    description = input()
                    response = agent.invoke({
                        "messages": [
                            {"role": "user", "content": "Plan update:"+description}
                        ]
                    },
                    config = config,
                    )

                print(f"\nReasoning behind the suggestion:")
                print(format_response(missing_plan["rationale"]))

            # Update synthesis request with new specifications
            synthesis_request = update_missing_handling_plan(synthesis_request, missing_plan)
            return handle_missing_values(df, synthesis_request)
        
#df, synthesis_request = handle_missing_values(df, synthesis_request)
#df.to_csv("../data/cat_texas_cleaned.csv", index=False)

Store information about the prepared dataset in JSON format for tool use (using the defined semantic types)

In [51]:
#csv_path = "../data/cat_texas_cleaned.csv"

def format_json(csv_path:str):
    df = pd.read_csv(csv_path)
    json_path = f"../data/{Path(csv_path).stem}.json"
    columns = []

    for col in df.columns:
        semantic_type = semantic_types[col]
        if semantic_type in ["Categorical", "Ordinal"]:
            values = sorted(df[col].dropna().astype(str).unique().tolist())

            columns.append({
                "name": col,
                "type": semantic_type,
                "i2s": values,
                "size": len(values)
                })

        elif semantic_type == "Float":
            columns.append({
                "name": col,
                "type": "Float",
                "min": float(df[col].min()),
                "max": float(df[col].max())
                })

        elif semantic_type == "Integer":
            columns.append({
                "name": col,
                "type": "Integer",
                "min": int(df[col].min()),
                "max": int(df[col].max())
                })

    metadata = {"columns": columns}

    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)
    return "JSON formatted for tool use."

#format_json(csv_path)

#### Recommend synthetic generation plan:

General selection plan:
- Numerical: CTGAN
  - Privacy: PATEGAN
- Categorical: BaysianNet
  - Privacy: PrivBayes
- Mix: CTGAN

---
Data generator tool information
- s3name = "" # Name of the dataset to run on {adult,census,credit,alarm,insurance}
- path = "" # Path to a local data file
- mechanism = ["PATEGAN", "CTGAN", "BayesianNet", "PrivBayes",] # {IndependentHistogram,BayesianNet,PrivBayes,CTGAN,PATEGAN}
- parameters = "" # The parameters of the mechanism to use separated by a colon
- output_file = "" # The file where to store the synthetic dataset
- sample_size = "" # The size of the synthetic dataset

In [52]:
# think of better name
def agent_g_synthesis(synthesis_request, semantic_types):
    prompt = """
You select and run a synthetic data generator.

Allowed mechanisms:
- PATEGAN
- CTGAN
- BayesianNet
- PrivBayes

Rules:
- If synthesis_request["generator"]["name"] is not "auto", use that generator.
- If privacy mode is differentially_private:
- mostly categorical data: PrivBayes
- mostly numerical or mixed data: PATEGAN
- If privacy mode is standard:
- mostly categorical data: BayesianNet
- mostly numerical data: CTGAN
- mixed data: CTGAN

Before calling the tool, briefly explain why the selected generator fits the data and use case.
Then call run_synthetic_generator.
"""
    # possibly include this human in the loop middleware at the beginning
    agent = create_agent(
        model=model,
        tools=[generator_tool],
        checkpointer=checkpointer,
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    generator_tool.name: {
                        "allowed_decisions": ["approve", "edit", "reject", "respond"]
                    }
                },
                description_prefix="Synthetic generator execution pending approval.",
            )
        ],
        system_prompt=prompt,
     )

    initial_input = {
        "messages": [
            {
                "role": "user",
                "content": f"""
    Current synthesis request:
    {synthesis_request}

    Semantic types:
    {semantic_types}

    Select a synthetic generator and explain why it would work best
    for the use case provided, then run the synthetic generator.
    """
            }
        ]
    }

    input_to_stream = initial_input
    while True:
        interrupted = False
        message_buffer = ""

        for chunk in agent.stream(input_to_stream, config=config, stream_mode=["updates", "messages"], version="v2",):
            if chunk["type"] == "messages":
                message = chunk["data"][0]
                content = getattr(message, "content", "")
                if content:
                    message_buffer += content

            if chunk["type"] == "updates" and "__interrupt__" in chunk["data"]:
                if message_buffer.strip():
                    print(message_buffer.strip().split("</think>")[1])

                print("\nINTERRUPTED:")
                #print(chunk["data"]["__interrupt__"])
                
                # To extract the suggested mechanism (if approved, added to synthesis_request)
                action_requests = chunk["data"]["__interrupt__"][0].value["action_requests"][0]

                interrupted = True
                break

        if not interrupted:
            break    

        decision = pyip.inputMenu(["approve", "edit", "respond", "reject"], numbered=True,)

        # Might help to build response nodes (e.g., approval, response, etc with no need for redefinition if used in later parts)
        if decision == "approve":
            resume_command = Command(
                resume={
                    "decisions": [
                        {"type": "approve"}
                    ]
                }
            )
            synthesis_request["generator"] = action_requests["args"]["mechanism"]

        elif decision == "edit":
            new_mechanism = pyip.inputMenu(["PATEGAN", "CTGAN", "BayesianNet", "PrivBayes",], "\nEnter replacement generator:\n", numbered=True,)

            resume_command = Command(
                resume={
                    "decisions": [
                        {
                            "type": "edit",
                            "edited_action": {
                                "name": generator_tool.name,
                                "args": {
                                    "mechanism": new_mechanism
                                },
                            },
                        }
                    ]
                }
            )
            synthesis_request["generator"] = new_mechanism

        elif decision == "respond":
            user_message = pyip.inputStr("\nWrite feedback regarding the data and tool choice:\n")
            resume_command = Command(
                resume={
                    "decisions": [
                        {
                            "type": "respond",
                            "message": user_message,
                        }
                    ]
                }
            )

        else:
            # In the basic human in the loop, but respond is likely sufficient
            resume_command = Command(
                resume={
                    "decisions": [
                        {
                            "type": "reject",
                            "message": "User rejected generator execution."
                        }
                    ]
                }
            )
        
        input_to_stream = resume_command
    return synthesis_request

#synthesis_request = agent_g_synthesis(synthesis_request, semantic_types)

In [53]:
# Updated with chosen generator
#synthesis_request

#### Generate data and report output:

In [54]:
def evaluation(real_df, synthetic_df, synthesis_request):
    generator_used = synthesis_request["generator"]
    jsd_results = jensen_shannon_distance(real=real_df, synthetic=synthetic_df)
    print("Jenson-Shannon Distances:")
    pprint.pprint(jsd_results)

    evaluation_prompt = ChatPromptTemplate.from_messages([
        ("system", """
    Evaluate whether synthetic data quality is satisfactory.

    Return exactly ONE valid JSON object.
    No markdown.
    No explanation outside of JSON.

    Interpretation:
    - Jensen-Shannon distance is between 0 and 1.
    - 0 means the distributions are identical.
    - average_jsd <= 0.10: satisfactory
    - 0.10 < average_jsd <= 0.25: acceptable but needs review
    - average_jsd > 0.25: not satisfactory
    - Columns with very high JSD (>0.50) are problem columns
    - whether the selected generator fits the data type

    JSON format:
    {{
    "evaluation": {{
        "status": "satisfactory",
        "average_jsd": 0.0,
        "problem_columns": [],
        "recommendation": "Brief recommendation",
        "rationale": "Brief explanation."
    }}
    }}
    """),
        ("user", """
    Synthesis request:
    {synthesis_request}

    Generator used:
    {generator_used}

    Jensen shannon distance results:
    {jsd_results}

    Evaluate whether the synthetic data is satisfactory.
    """)
    ])

    chain = evaluation_prompt | model # using the model directly here (agent is not making JSON)
    plan = chain.invoke({
        "synthesis_request": synthesis_request,
        "generator_used": generator_used,
        "jsd_results": jsd_results,
        },
        config = config,
        )

    

    plan = plan.content.split("</think>")[1].strip()
    parsed = json.loads(plan)
    evaluation = parsed["evaluation"]

    print("\nEvaluation:")
    print(f"- Status: {evaluation['status']}")
    print(f"- Average Jenson-Shannon distance: {evaluation['average_jsd']}")
    print(f"- Problem columns: {evaluation['problem_columns']}")
    print(f"- Recommendation:")#\n{evaluation['recommendation']}")
    format_response(evaluation["recommendation"])

    # Agent plan for improvement
    # Human input on the results and evaluation
    # TODO: Add confirmation from user to retry if seen as necessary
    # If not satisfactory (agent based), prompt agent again but to suggest
    #   what changes should be made to the synthesis request
#evaluation(real_df, synth_df, synthesis_request)

Other metrics from Syndat:

In [55]:
# Not the best in a mostly categorical dataset
#print(normalized_correlation_difference(real=real_df, synthetic=synth_df, ignore_cat=True))

In [56]:
#print(discriminator_auc(real=real_df, synthetic=synth_df))

---

---

In [59]:
# Select preferences
synthesis_request = select_preferences()

# Upload and clean data
df = upload_data(data_path, synthesis_request["data_scope"]["selected_columns"])
df = handle_dtypes(df)
df, semantic_types = handle_semantic_types(df)
df, synthesis_request = handle_missing_values(df, synthesis_request)
df.to_csv("../data/cat_texas_cleaned.csv", index=False)

# JSON
csv_path = "../data/cat_texas_cleaned.csv"
format_json(csv_path)

# Synthesize data
synthesis_request = agent_g_synthesis(synthesis_request, semantic_types)
real_df = pd.read_csv("../data/cat_texas_after_imputation.csv")
synth_df = pd.read_csv("../output/synthetic_cat_texas_after_imputation_pategan.csv").iloc[:,1:]

# Evaluate data
evaluation(real_df, synth_df, synthesis_request)

What would you like to use the synthetic data for?
Please select one of the following:
1. basic generation
2. predictive utility

Would you like to generate the same amount of samples as the dataset or a fixed number of samples?
Please select one of the following:
1. fixed value
2. same as original

How many synthetic samples would you like to generate?

Are you interested in differential privacy?

Is there a specific data generator that you want to use on your data?
	You will receive a recommendation based on the dataset's characteristics.

How would you like missing values to be handled? Select 'auto' if you would like a suggestion.
Please select one of the following:
1. impute
2. threshold
3. remove
4. auto

-------------------------Your preferences have been recorded-------------------------
{'data_scope': {'selected_columns': []},
 'generator': {'name': 'auto'},
 'missing_value_handling': {'impute': {'categorical_strategy': None,
                                       'method': No